# ISLES2024 — Final 149-patient paired four-metric evaluation (Dataset504 vs Dataset505)

**Runtime → Change runtime type → CPU (no GPU needed) → Run all.**

This notebook computes the remaining **lesion-wise F1** and **absolute lesion-count difference** for **NCCT + Tmax + CBF + CBV + MTT** with the repository's original **Panoptica 0.2-threshold** implementation. It also recomputes Dice and absolute volume difference from NIfTI masks, cross-checks per-patient nnU-Net Dice, and performs the **four-metric paired comparison against the frozen 149-patient NCCT-only baseline**. It performs **no retraining**.

It does not use unavailable organizer-specific NCCT brain masks. Therefore all outputs are **local patient-wise out-of-fold** metrics, **not official ISLES'24 hidden-test scores**.

### Scientific integrity

- Patient-wise split identity and SHA256 verified before evaluation; all 149 validation patients occur exactly once.
- Fail-closed prediction/reference geometry checks, checkpoint presence, and exact fold membership.
- Persistent per-patient case checkpoints and progress logs allow **resume after Colab disconnection**.
- Four predetermined endpoints, 20,000 paired patient bootstrap replicates, 100,000 sign-flip permutation replicates, and **Holm multiplicity correction** across the four tests.
- A frozen repository commit is used for evaluator consistency. Detailed outputs stay in your private Google Drive.

Original evaluator: https://github.com/ezequieldlrosa/isles24/blob/6b4dea6f07bf1717cca638ea1fe9345a47c3cd41/utils/eval_utils.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Prepare output folders and pinned source (CPU only)

A **separate** `perfusion_oof_evaluation` output directory is used, protecting the frozen Dataset504 baseline. Progress is logged to Google Drive.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, zipfile

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
PREPROCESSED = ROOT / 'nnUNet_preprocessed'
RESULTS = ROOT / 'nnUNet_results'
PERF_EVAL = ROOT / 'results' / 'perfusion_oof_evaluation'
PAIRED = ROOT / 'results' / 'ncct_vs_perfusion_paired'
BASELINE = ROOT / 'results' / 'ncct_oof_evaluation' / 'isles2024_ncct_oof_case_metrics.csv'
LOG = PERF_EVAL / 'perfusion_oof_resumable.log'
REPO_COMMIT = '40e29c2e5037cfef42c2a499010a8c2a81f0e0a3'
ZIP = Path('/content/stroke_eval_pinned.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-' + REPO_COMMIT)
PERF_EVAL.mkdir(parents=True, exist_ok=True)
PAIRED.mkdir(parents=True, exist_ok=True)

ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + ENV.get('PYTHONPATH', '')
ENV['PYTHONUNBUFFERED'] = '1'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fp:
        for block in iter(lambda: fp.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def run_logged(label, command):
    command = [str(v) for v in command]
    print('\n===', label, '===\n$', ' '.join(command), flush=True)
    with LOG.open('a', encoding='utf8') as lf:
        lf.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {label}: $ {' '.join(command)}\n")
        lf.flush()
        with subprocess.Popen(command, cwd=REPO if REPO.exists() else None, env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, bufsize=1, text=True) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                lf.write(line)
                lf.flush()
            return_code = proc.wait()
        lf.write(f'[{label}] exit_code={return_code}\n')
        lf.flush()
    if return_code != 0:
        raise RuntimeError(f'{label} failed (exit code {return_code}); inspect {LOG}')

for directory in (PREPROCESSED, RESULTS):
    if not directory.is_dir(): raise FileNotFoundError(directory)
if not BASELINE.is_file(): raise FileNotFoundError(f'Frozen NCCT baseline missing: {BASELINE}')

if not (REPO / 'scripts/evaluate_isles2024_nnunet_oof.py').is_file():
    run_logged('Download pinned evaluator', ['curl', '-fLsS', '--retry', '5', '--retry-delay', '4',
        f'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/{REPO_COMMIT}.zip', '-o', ZIP])
    with zipfile.ZipFile(ZIP) as z: z.extractall('/content')
if not (REPO / 'scripts/evaluate_isles2024_nnunet_oof.py').is_file():
    raise RuntimeError('Expected evaluator not found in pinned repository archive')
print('Pinned evaluator commit:', REPO_COMMIT)
print('Persistent progress:', PERF_EVAL / 'ncct_oof_progress.json')
print('Persistent log:', LOG)


## 2. Verify frozen folds and 149 prediction masks

This check fails immediately if the dataset split, training checkpoints, cohort coverage, or prediction inventory is incomplete. No evaluation begins on inconsistent data.


In [ ]:
from collections import Counter
sys.path.insert(0, str(REPO / 'src'))
from hybridstrokeseg.isles2024_eval import load_splits, validate_oof_splits

NCCT_PREP = PREPROCESSED / 'Dataset504_ISLES2024_NCCT'
PERF_PREP = PREPROCESSED / 'Dataset505_ISLES2024_NCCT_PERFUSION'
PERF_TRAIN = (RESULTS / 'Dataset505_ISLES2024_NCCT_PERFUSION' /
              'nnUNetTrainer__nnUNetPlans__3d_fullres')
split_ncct = NCCT_PREP / 'splits_final.json'
split_perf = PERF_PREP / 'splits_final.json'
expected_sha = '664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(split_ncct) == expected_sha, 'Frozen Dataset504 split hash mismatch'
assert sha256(split_perf) == expected_sha, 'Dataset505 split differs from frozen baseline'
splits = load_splits(split_perf)
assert len(splits) == 5
cases = validate_oof_splits(splits)
assert len(cases) == 149
assert [len(s['val']) for s in splits] == [30,30,30,30,29]
for fold, sp in enumerate(splits):
    fold_dir = PERF_TRAIN / f'fold_{fold}'
    ckpt = fold_dir / 'checkpoint_final.pth'
    masks = sorted(p.name.removesuffix('.nii.gz') for p in (fold_dir / 'validation').glob('*.nii.gz'))
    assert ckpt.is_file() and ckpt.stat().st_size > 0, f'Fold {fold} has no trained final checkpoint'
    assert masks == sorted(sp['val']), f'Fold {fold} validation prediction/membership mismatch'
    assert (fold_dir / 'validation' / 'summary.json').is_file(), f'Fold {fold} lacks validation summary'
    print(f'Fold {fold}: {len(masks)} masks, checkpoint {ckpt.stat().st_size:,} bytes — PASS')
print('149-patient complete OOF coverage / exact split SHA256 — PASS')


## 3. Install the narrow evaluation dependencies and run a synthetic metric smoke test

This reuses the **wheel-only dependency strategy** of the successful NCCT-only CPU evaluator notebook. Do not install the complete nnU-Net training environment.


In [ ]:
run_logged('Install wheel-only medical imaging metric dependencies', [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--only-binary=:all:',
    'nibabel>=5.2,<6', 'connected-components-3d>=3.12,<4', 'surface-distance>=0.1,<1',
    'ruamel.yaml>=0.18,<1', 'scikit-image>=0.23,<1', 'SimpleITK>=2.3,<3',
    'rich>=13,<15', 'pandas>=2.2,<3', 'plotly>=5,<7'])
run_logged('Install pinned Panoptica 1.4.1 wheel', [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--only-binary=:all:', '--no-deps', 'panoptica==1.4.1'])
import importlib.util
if importlib.util.find_spec('torch') is None:
    run_logged('Install PyTorch CPU wheel', [sys.executable, '-m', 'pip', 'install',
        '--only-binary=:all:', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
smoke = r"""
import numpy as np
from importlib.metadata import version
from hybridstrokeseg.isles2024_eval import evaluate_case
mask=np.zeros((7,7,7),dtype=np.uint8); mask[2:5,2:5,2:5]=1
x=evaluate_case(case_id='synthetic',fold=0,reference=mask,prediction=mask.copy(),voxel_volume_ml=0.001)
assert abs(x.dice-1)<1e-8 and abs(x.lesion_f1-1)<1e-8, x
print('Panoptica API synthetic smoke PASS:',version('panoptica'))
"""
run_logged('Pinned Panoptica smoke test', [sys.executable, '-c', smoke])


## 4. Compute **all four** Dataset505 OOF metrics, resumably

Evaluation reads the stored 149 prediction and reference NIfTI files; it neither modifies the models nor generates new predictions. Each validated case is cached with its input signature; rerunning after a disconnect resumes safely.

Expect separate local Dice, absolute volume difference (mL), lesion-wise F1, and absolute lesion-count difference values.


In [ ]:
run_logged('Dataset505 149-case four-metric local OOF evaluation', [
    sys.executable, '-u', 'scripts/evaluate_isles2024_nnunet_oof.py',
    '--dataset-id', '505', '--configuration', '3d_fullres',
    '--preprocessed-root', PREPROCESSED,
    '--results-root', RESULTS,
    '--output-dir', PERF_EVAL,
    '--bootstrap-repetitions', '10000', '--bootstrap-seed', '2026'])

# The legacy evaluator's filename has 'ncct' hard-coded. Copy, never overwrite,
# into unmistakably Dataset505-named outputs after all cases have completed.
for suffix in ('case_metrics.csv','summary.json','manifest.json'):
    old = PERF_EVAL / f'isles2024_ncct_oof_{suffix}'
    new = PERF_EVAL / f'isles2024_perfusion_oof_{suffix}'
    if not old.is_file(): raise FileNotFoundError(old)
    shutil.copy2(old, new)
print('Dataset505 case metrics:', PERF_EVAL / 'isles2024_perfusion_oof_case_metrics.csv')


## 5. Four-metric paired inference with Holm correction

The included analysis script enforces exact 149-patient pairing, fold identity, voxel-volume and reference-volume equality, and unmasked-label comparability. It saves a detailed per-patient CSV, JSON results, and a Markdown report to `results/ncct_vs_perfusion_paired`.


In [ ]:
from pathlib import Path
import hashlib
PAIR_SCRIPT = Path('/content/isles2024_paired_analysis.py')
PAIRED_SCRIPT_SOURCE = r'''#!/usr/bin/env python3
"""Fail-closed paired ISLES'24 NCCT versus NCCT+perfusion local OOF analysis.

There are two modes:
  --perfusion-summaries <fold0.json> ... <fold4.json>
      Validates five nnU-Net summaries against the frozen NCCT case-level CSV;
      computes voxel Dice and absolute volume difference only. Lesion metrics
      are intentionally unavailable in this mode (need mask-level Panoptica).
  --perfusion-csv <Dataset505 OOF case metrics CSV>
      Validates 149 paired OOF cases, frozen folds and reference geometry;
      computes all four paired endpoints with bootstrap, permutation p-values,
      and Holm multiplicity correction.

Primary sign convention is perfusion minus NCCT; negative difference is better
for absolute errors, positive difference is better for Dice and lesion-F1.
These are *local out-of-fold*, not official hidden-test challenge metrics.
"""
from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np

METRICS = (
    ("dice", "Dice", "higher"),
    ("absolute_volume_difference_ml", "Absolute volume difference (mL)", "lower"),
    ("lesion_f1", "Lesion-wise F1", "higher"),
    ("absolute_lesion_count_difference", "Absolute lesion-count difference", "lower"),
)
PERF_FIELDS = ("dice", "absolute_volume_difference_ml", "lesion_f1", "absolute_lesion_count_difference")
REQUIRED = {"case_id", "fold", "dice", "absolute_volume_difference_ml", "reference_volume_ml", "voxel_volume_ml"}


def load_csv(path: Path, *, expected_n: int) -> dict[str, dict]:
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open(newline="", encoding="utf8") as handle:
        reader = csv.DictReader(handle)
        if not REQUIRED.issubset(reader.fieldnames or ()):
            raise ValueError(f"{path}: missing fields {sorted(REQUIRED - set(reader.fieldnames or []))}")
        rows = list(reader)
    unique = {r["case_id"]: r for r in rows}
    if len(rows) != expected_n or len(unique) != expected_n:
        raise ValueError(f"{path}: require {expected_n} unique cases; found {len(rows)} rows, {len(unique)} unique")
    if any(not k.startswith("sub_stroke") for k in unique):
        raise ValueError(f"{path}: unexpected patient identifiers")
    if any((r.get("brain_mask_path") or "").strip() for r in rows):
        raise ValueError(f"{path}: brain-masked results cannot be compared to unmasked frozen baseline")
    return unique


def load_perfusion_summaries(paths: list[Path], baseline: dict[str, dict], expected_n: int) -> dict[str, dict]:
    if len(paths) != 5:
        raise ValueError("Provide exactly five nnU-Net fold validation summary.json paths in fold 0..4 order")
    out: dict[str, dict] = {}
    for fold, path in enumerate(paths):
        obj = json.loads(path.read_text(encoding="utf8"))
        for row in obj.get("metric_per_case", []):
            source = Path(row["prediction_file"])
            if source.suffixes[-2:] != [".nii", ".gz"]:
                raise ValueError(f"Malformed summary prediction path: {source}")
            case_id = source.name.removesuffix(".nii.gz")
            if case_id in out:
                raise ValueError(f"Duplicate Dataset505 prediction: {case_id}")
            if case_id not in baseline:
                raise ValueError(f"Unmatched Dataset505 case: {case_id}")
            old = baseline[case_id]
            voxel_ml = float(old["voxel_volume_ml"])
            m = row["metrics"]["1"]
            if not all(np.isfinite(float(m[k])) for k in ("Dice", "n_pred", "n_ref")):
                raise ValueError(f"Non-finite Dataset505 metrics: {case_id}")
            gt_505_volume_ml = float(m["n_ref"]) * voxel_ml
            old_gt_volume_ml = float(old["reference_volume_ml"])
            if not np.isclose(gt_505_volume_ml, old_gt_volume_ml, atol=1e-5, rtol=1e-6):
                raise ValueError(
                    f"{case_id}: reference volume differs between Dataset504/505: "
                    f"{old_gt_volume_ml} vs {gt_505_volume_ml} ml"
                )
            out[case_id] = {
                "case_id": case_id,
                "fold": str(fold),
                "dice": repr(float(m["Dice"])),
                "absolute_volume_difference_ml": repr(abs(float(m["n_pred"]) - float(m["n_ref"])) * voxel_ml),
                "reference_volume_ml": repr(gt_505_volume_ml),
                "voxel_volume_ml": repr(voxel_ml),
            }
    if len(out) != expected_n:
        raise ValueError(f"Expected {expected_n} unique Dataset505 cases, found {len(out)}")
    return out


def validate_pairs(baseline: dict[str, dict], perfusion: dict[str, dict], *, full: bool) -> list[dict]:
    if baseline.keys() != perfusion.keys():
        raise ValueError(f"Patient sets differ: missing={sorted(baseline.keys()-perfusion.keys())}, extra={sorted(perfusion.keys()-baseline.keys())}")
    paired = []
    for case_id in sorted(baseline):
        a, b = baseline[case_id], perfusion[case_id]
        if int(a["fold"]) != int(b["fold"]):
            raise ValueError(f"Patient-wise fold mismatch for {case_id}")
        for attr, atol in (("reference_volume_ml", 1e-5), ("voxel_volume_ml", 1e-9)):
            if not np.isclose(float(a[attr]), float(b[attr]), atol=atol, rtol=1e-6):
                raise ValueError(f"{case_id}: {attr} is different across modalities")
        if full and "reference_lesion_count" in a and "reference_lesion_count" in b:
            if int(a["reference_lesion_count"]) != int(b["reference_lesion_count"]):
                raise ValueError(f"{case_id}: reference connected-component count differs")
        row = {"case_id": case_id, "fold": int(a["fold"]), "reference_volume_ml": float(a["reference_volume_ml"])}
        for key, _, _ in METRICS[:4 if full else 2]:
            if key not in b:
                raise ValueError(f"{case_id}: Dataset505 metric missing: {key}")
            old, new = float(a[key]), float(b[key])
            if not np.isfinite(old) or not np.isfinite(new):
                raise ValueError(f"Non-finite result for {case_id}/{key}")
            if key in ("dice", "lesion_f1") and not (0 <= old <= 1 and 0 <= new <= 1):
                raise ValueError(f"Invalid score outside [0,1]: {case_id}/{key}")
            if key not in ("dice", "lesion_f1") and (old < 0 or new < 0):
                raise ValueError(f"Negative absolute error for {case_id}/{key}")
            row[f"ncct_{key}"] = old
            row[f"perfusion_{key}"] = new
            row[f"delta_{key}"] = new - old
        paired.append(row)
    fold_sizes = {f: sum(x["fold"] == f for x in paired) for f in range(5)}
    if fold_sizes != {0: 30, 1: 30, 2: 30, 3: 30, 4: 29}:
        raise ValueError(f"Frozen folds do not match 30/30/30/30/29: {fold_sizes}")
    return paired


def percentile_ci(values: np.ndarray, *, n_reps: int, seed: int) -> list[float]:
    rng = np.random.default_rng(seed)
    # Chunked to avoid >GB RAM allocations at large bootstrap counts.
    means = []
    for batch_start in range(0, n_reps, 2000):
        batch_n = min(2000, n_reps-batch_start)
        idx = rng.integers(0, len(values), size=(batch_n, len(values)))
        means.append(values[idx].mean(axis=1))
    return [float(x) for x in np.quantile(np.concatenate(means), [0.025, 0.975])]


def paired_permutation_p(delta: np.ndarray, *, n_reps: int, seed: int) -> float:
    observed = abs(float(delta.mean()))
    rng = np.random.default_rng(seed)
    tail = 0
    for batch_start in range(0, n_reps, 2000):
        batch_n = min(2000, n_reps - batch_start)
        flips = rng.integers(0, 2, size=(batch_n, len(delta)), dtype=np.int8) * 2 - 1
        dist = np.abs(flips @ delta / len(delta))
        tail += int(np.count_nonzero(dist >= observed - 1e-12))
    return float((tail + 1) / (n_reps + 1))


def holm_adjust(pvalues: list[float]) -> list[float]:
    """Holm step-down family-wise adjusted p-values."""
    n = len(pvalues)
    order = np.argsort(pvalues)
    adj = np.empty(n)
    current = 0.0
    for rank, original_index in enumerate(order):
        current = max(current, (n-rank)*pvalues[int(original_index)])
        adj[int(original_index)] = min(current, 1.0)
    return [float(x) for x in adj]


def analyze(paired: list[dict], *, full: bool, n_boot: int, n_perm: int, seed: int) -> dict:
    n = len(paired)
    metrics = METRICS if full else METRICS[:2]
    result = {"n_patients": n, "fold_n": {str(f): sum(x["fold"] == f for x in paired) for f in range(5)},
              "mode": "four_metrics_full" if full else "dice_volume_interim", "metrics": {},
              "note": "Local matched out-of-fold validation, not the official ISLES'24 hidden-test evaluation.",
              "bootstrap_repetitions": n_boot, "permutation_repetitions": n_perm, "rng_seed": seed,
              "brain_mask_applied": False, "multiple_testing_method": "Holm family-wise correction (only in full four-metric mode)"}
    pvals = []
    for idx, (key, human, direction) in enumerate(metrics):
        base = np.asarray([r[f"ncct_{key}"] for r in paired], dtype=float)
        perf = np.asarray([r[f"perfusion_{key}"] for r in paired], dtype=float)
        delta = perf - base
        p = paired_permutation_p(delta, n_reps=n_perm, seed=seed+100+idx)
        pvals.append(p)
        result["metrics"][key] = {
            "name": human, "better": direction, "n": n,
            "ncct_mean": float(base.mean()), "perfusion_mean": float(perf.mean()),
            "perfusion_minus_ncct_mean": float(delta.mean()),
            "delta_95ci_paired_bootstrap": percentile_ci(delta, n_reps=n_boot, seed=seed+idx),
            "two_sided_sign_flip_permutation_p_nominal": p,
            "n_improved": int(((perf > base) if direction == "higher" else (perf < base)).sum()),
            "n_worsened": int(((perf < base) if direction == "higher" else (perf > base)).sum()),
            "n_unchanged": int((perf == base).sum()),
            "fold_means": {
                str(f): {"n": sum(r["fold"]==f for r in paired),
                         "ncct": float(np.mean([r[f"ncct_{key}"] for r in paired if r["fold"] == f])),
                         "perfusion": float(np.mean([r[f"perfusion_{key}"] for r in paired if r["fold"] == f]))}
                for f in range(5)
            },
        }
    if full:
        adjusted = holm_adjust(pvals)
        for i,(key,_,_) in enumerate(metrics):
            result["metrics"][key]["holm_adjusted_p_4_metrics"] = adjusted[i]
        result["four_metric_lesion_count_statistics_available"] = True
    else:
        result["four_metric_lesion_count_statistics_available"] = False
        result["pending_evaluation"] = ["lesion_f1", "absolute_lesion_count_difference"]
        result["caution"] = "Dice and volume p-values are nominal exploratory tests; four-endpoint Holm correction pending mask-level evaluation."
    result["reference_volume_quartiles_ml"] = [float(v) for v in np.quantile([r["reference_volume_ml"] for r in paired], [0.25,0.5,0.75])]
    return result


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--baseline-csv", type=Path, required=True)
    mx = ap.add_mutually_exclusive_group(required=True)
    mx.add_argument("--perfusion-csv", type=Path)
    mx.add_argument("--perfusion-summaries", type=Path, nargs=5, metavar="JSON")
    ap.add_argument("--output-dir", type=Path, required=True)
    ap.add_argument("--bootstrap-repetitions", type=int, default=20_000)
    ap.add_argument("--permutation-repetitions", type=int, default=100_000)
    ap.add_argument("--seed", type=int, default=20261119)
    ap.add_argument("--expected-n", type=int, default=149)
    args = ap.parse_args()
    if args.expected_n != 149:
        raise ValueError("Study protocol requires all 149 patients; --expected-n must remain 149")
    if args.bootstrap_repetitions < 100 or args.permutation_repetitions < 100:
        raise ValueError("At least 100 resamples required")
    baseline = load_csv(args.baseline_csv, expected_n=args.expected_n)
    full = args.perfusion_csv is not None
    if full:
        perfusion = load_csv(args.perfusion_csv, expected_n=args.expected_n)
        required_full = {"lesion_f1", "absolute_lesion_count_difference"}
        if any(not required_full.issubset(r) for r in perfusion.values()):
            raise ValueError("Dataset505 full OOF output lacks lesion-wise metrics")
    else:
        perfusion = load_perfusion_summaries(args.perfusion_summaries, baseline, args.expected_n)
    paired = validate_pairs(baseline, perfusion, full=full)
    summary = analyze(paired, full=full, n_boot=args.bootstrap_repetitions, n_perm=args.permutation_repetitions, seed=args.seed)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    pathbase = "isles2024_paired_full" if full else "isles2024_paired_dice_volume"
    with (args.output_dir / f"{pathbase}_cases.csv").open("w", newline="", encoding="utf8") as fp:
        writer = csv.DictWriter(fp, fieldnames=list(paired[0]))
        writer.writeheader(); writer.writerows(paired)
    (args.output_dir / f"{pathbase}_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf8")
    lines = ["# ISLES2024: paired Dataset504 vs Dataset505 evaluation", "", "**Scope:** matched, local five-fold out-of-fold validation (149 patients); not the official hidden-test challenge.", "", f"**Mode:** {'Four-metric mask-level evaluation' if full else 'Interim Dice and volume analysis'}.", "", "| Metric | NCCT | NCCT + perfusion | Δ (perfusion − NCCT) | 95% paired CI | Sign-flip p (nominal) |", "|---|---:|---:|---:|---|---:|"]
    for k, _, _ in (METRICS if full else METRICS[:2]):
        x = summary["metrics"][k]
        ci = x["delta_95ci_paired_bootstrap"]
        lines.append(f"| {x['name']} | {x['ncct_mean']:.5f} | {x['perfusion_mean']:.5f} | {x['perfusion_minus_ncct_mean']:+.5f} | [{ci[0]:+.5f}, {ci[1]:+.5f}] | {x['two_sided_sign_flip_permutation_p_nominal']:.5f} |")
    lines.extend(["", "## Scientific interpretation", "",
      "- **Study design:** identical patient-wise folds, no patient leakage; all 149 paired cases have identical ground-truth lesion volumes.",
      "- **Directions:** Dice and lesion-F1 higher is better; absolute volume and lesion-count errors lower is better.",
      "- **Uncertainty:** 95% paired patient bootstrap intervals, two-sided patient sign-flip permutation p-values.",
      "- **Multiplicity:** " + ("Holm adjustment was computed across all four prespecified metrics; see JSON for adjusted p-values." if full else "Only two metrics are available. Significance is exploratory; the other two prespecified lesion metrics require mask-level Panoptica evaluation."),
      "- **Masking:** organizer-specific NCCT-derived brain masks were not applied; these are local OOF metrics, not official ISLES challenge test scores.",
      "- **Heterogeneity:** patient-level responses differ; an overall mean gain does not imply every patient benefits.",
      ""])
    (args.output_dir / f"{pathbase}_report.md").write_text("\n".join(lines), encoding="utf8")
    print("Verified", len(paired), "paired cases; all five frozen folds.")
    print("Mode:", summary["mode"])
    for k, obj in summary["metrics"].items():
        print(f"{k}: NCCT={obj['ncct_mean']:.6f}, NCCT+perfusion={obj['perfusion_mean']:.6f}, Δ={obj['perfusion_minus_ncct_mean']:+.6f}, 95% CI={obj['delta_95ci_paired_bootstrap']}, nominal p={obj['two_sided_sign_flip_permutation_p_nominal']:.5f}, adjusted p={obj.get('holm_adjusted_p_4_metrics')}")
    print("Outputs:", args.output_dir)


if __name__ == "__main__":
    main()
'''
PAIR_SCRIPT.write_text(PAIRED_SCRIPT_SOURCE, encoding='utf8')
expected_hash = '1d96285b6015d486da725e6af5fd3d2b38507b51ea860d3ee8c6ecf6efbb9233'
assert hashlib.sha256(PAIR_SCRIPT.read_bytes()).hexdigest() == expected_hash
print('Paired analysis script SHA256:', expected_hash)
run_logged('149-patient complete paired four-metric analysis', [
    sys.executable, '-u', PAIR_SCRIPT,
    '--baseline-csv', BASELINE,
    '--perfusion-csv', PERF_EVAL / 'isles2024_perfusion_oof_case_metrics.csv',
    '--output-dir', PAIRED,
    '--bootstrap-repetitions', '20000', '--permutation-repetitions', '100000',
    '--seed', '20261119'])
provenance = {
    'evaluator_github_commit': REPO_COMMIT,
    'evaluator_source_sha256': sha256(REPO/'src/hybridstrokeseg/isles2024_eval.py'),
    'evaluator_script_sha256': sha256(REPO/'scripts/evaluate_isles2024_nnunet_oof.py'),
    'paired_script_sha256': expected_hash,
    'frozen_split_sha256': expected_sha,
    'baseline_file_sha256': sha256(BASELINE),
    'perfusion_case_metrics_sha256': sha256(PERF_EVAL/'isles2024_perfusion_oof_case_metrics.csv'),
    'brain_mask_applied': False,
    'n_oof_patients': 149,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
}
(PAIRED/'isles2024_paired_provenance.json').write_text(json.dumps(provenance, indent=2)+'\n')
print('\nFINAL COMPARISON:', PAIRED/'isles2024_paired_full_report.md')
print('JSON RESULTS:', PAIRED/'isles2024_paired_full_summary.json')
print('PAIRED PATIENT DATA:', PAIRED/'isles2024_paired_full_cases.csv')


## 6. Interpret results carefully

For **Dice** and **lesion-wise F1**, positive `perfusion_minus_ncct_mean` favors perfusion. For **absolute volume difference** and **absolute lesion-count difference**, negative values favor perfusion. Always quote both the paired 95% CI and **Holm-adjusted** p-values for the four metrics; an unadjusted p-value near 0.03 need not remain below 0.05 after correction. This is an **internal dataset ablation**, not an organizer-ranked hidden-test score.

If Colab stops mid-evaluation, rerun **Run all**. Existing validated case checkpoints remain in Google Drive and are reused when their inputs have not changed.
